In [ ]:
import os
import pandas as pd
import pmdarima as pm
from pmdarima import model_selection
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import numpy as np
import warnings
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

In [ ]:
for i in ids:
    ts = patients[i].loc[:, "glucose"]
    train_data = ts.iloc[:-200]
    test_data = ts.iloc[-200:]
    tscv = TimeSeriesSplit(n_splits=5, test_size=10)

    # Store the Mean Squared Errors (MSE) from each fold
    mse_scores = []
    fold_results = []
    f = 1

    # Convert the pandas Series to a numpy array for splitting
    X = train_data.values

    # Iterate through the splits
    for train_index, test_index in tscv.split(X):
        # Split the data
        cv_train = X[train_index]
        cv_test = X[test_index]


        cv_model = pm.ARIMA(order=(4,0,3))
        cv_model.fit(cv_train)

        # --- Make Predictions on the Test Fold ---
        # The 'n_periods' is the length of the test set
        forecast, conf_int = cv_model.predict(n_periods=len(cv_test), return_conf_int=True)

        # --- Evaluate ---
        mse = mean_squared_error(cv_test, forecast)
        mse_scores.append(mse)

        # Store results for plotting
        fold_results.append({
            'train_data': train_data.iloc[train_index],
            'test_data': train_data.iloc[test_index],
            'forecast': pd.Series(forecast, index=train_data.index[test_index])
        })

        print(f"Fold {f} MSE: {mse:.2f}")
        f += 1

    # Calculate and print the overall cross-validation score
    avg_mse = np.mean(mse_scores)
    print(f"\nAverage Cross-Validation MSE: {avg_mse:.2f}")